# Task 4.5. Baseline mô hình text pretrained

| | |
|---|---|
| **Mục đích** | Đo xem mô hình ngôn ngữ đã pretrained trên mã nguồn mạnh tới đâu trên đúng split của nhóm, làm mốc cho mọi kết quả GNN của Phase 4 |
| **Đầu vào** | Kaggle Dataset chứa tệp `diversevul_text.jsonl.gz` do `scripts/build_text_dataset.py` sinh ra |
| **Đầu ra** | `results_4_5/<MODEL>/`, `figures_4_5/`, thư mục `submission_4_5` cùng tệp zip để nộp |
| **Máy chạy** | Kaggle GPU T4 hoặc P100, bật Internet để tải trọng số pretrained |

**Thiết kế khác mọi task trước.** 4.1b và 4.2 là bài toán *chọn cấu hình*. 4.3 là bài toán *đo tác động*. 4.5 là một *baseline*: cấu hình được chốt một lần từ bài báo, chạy đủ hai protocol với ba seed, test báo cáo cho đúng cấu hình đó. Không dò siêu tham số, vì câu hỏi là "mô hình text pretrained mạnh tới đâu", không phải "cấu hình text nào tốt nhất".

**Dữ liệu cũng khác.** Mô hình text cần mã nguồn gốc. Kaggle Dataset `DiverseVul 9077 - Processed Graph Data` chỉ chứa id token theo vocabulary tự học nên không dùng được ở đây. Tệp input của notebook này được dựng riêng từ jsonl DiverseVul thô, manifest split của 2.3 và danh sách mẫu lỗi của 3.3, giữ đúng 7.799 mẫu hợp lệ mà các model graph đã dùng.

| Lựa chọn | Nguồn |
|---|---|
| Cắt hàm dài thành các đoạn 512 token rồi max-pooling | EMNLP 2024, bài 1 trong tab Tài liệu tham khảo |
| CodeBERT là model chính | Bài 1 báo CodeBERT tốt nhất trong ba model họ thử |
| Learning rate 2e-5, tối đa 10 epoch | PrimeVul, bài 3 |
| Class-weighted cross-entropy, 23 CWE, seed 42 43 44, hai protocol | Quy ước Task 4.1a |

Những giá trị còn lại, như batch size, warmup và gradient clipping, là thông lệ fine-tune transformer chứ không trích từ bài báo. Bài 1 không công bố các giá trị này.

**Không so trực tiếp với con số trong bài báo.** Bài 1 báo accuracy và weighted-F1 trên Big-Vul với split ngẫu nhiên. Nhóm dùng DiverseVul, split chống rò rỉ theo project, metric chính là Macro-F1. Chỉ so trong nội bộ, qua các baseline chạy trên cùng split.

In [ ]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
APPROVED_BRANCH = "feature/4_5_text_baseline"
APPROVED_COMMIT = "2a81f91e585a7948defb4b9e238c3364db173885"  # SHA đầy đủ của commit chứa mã nguồn
MODEL = "codebert"  # model chính; đổi sang "unixcoder" cho lần chạy thứ hai nếu còn thời gian
SECOND_MODEL = None  # ví dụ "unixcoder"; để None nếu không đủ thời gian
RUN_EXPERIMENTS = True
ACTIVE = RUN_EXPERIMENTS  # các ô chuẩn bị: clone, kiểm tra dữ liệu, thống kê độ dài
# Phục hồi sau khi phiên chết: gắn output cũ làm dataset rồi trỏ tới thư mục kết quả đã lưu.
SAVED_RESULTS_ROOT = None  # ví dụ /kaggle/input/<saved-output>/results_4_5
OUT_DIR = Path("/kaggle/working/results_4_5")
FIG_DIR = Path("/kaggle/working/figures_4_5")
SUBMISSION_DIR = Path("/kaggle/working/submission_4_5")
if RUN_EXPERIMENTS:
    if not Path("/kaggle/input").is_dir():
        raise RuntimeError("Chỉ chạy trên Kaggle với dataset chính thức.")
    if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
        raise ValueError("Cần SHA đầy đủ đã được phê duyệt.")
REPO_DIR = Path("/tmp/g45") / APPROVED_COMMIT
READ_ROOT = Path(SAVED_RESULTS_ROOT) if SAVED_RESULTS_ROOT else OUT_DIR
MODELS_RUN = [m for m in (MODEL, SECOND_MODEL) if m]

## 1. Tải mã nguồn và ghi phiên bản

In [ ]:
if ACTIVE:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", "--single-branch", "--branch",
                        APPROVED_BRANCH, REPO_URL, str(REPO_DIR)], check=True, timeout=180)
    subprocess.run(["git", "checkout", "--detach", APPROVED_COMMIT], cwd=REPO_DIR, check=True)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    assert actual == APPROVED_COMMIT
    sys.path.insert(0, str(REPO_DIR))
    import torch
    from src.training import text_experiments as tx
    from src.training.gcn_tuning import code_identity, environment
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings (T4 hoặc P100)."
    assert torch.ones(1, device="cuda").sum().item() == 1, "Không khởi tạo được GPU."
    assert not code_identity(REPO_DIR)["dirty"], "Checkout phải sạch."
    assert set(MODELS_RUN) <= set(tx.MODELS), f"Model phải nằm trong {sorted(tx.MODELS)}"
    try:
        import transformers
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
        import transformers
    print("Revision:", actual, flush=True)
    print("Environment:", environment(), "transformers", transformers.__version__, flush=True)


def run_script(script, arguments, log_path):
    cmd = [sys.executable, script, *arguments]
    print("Command:", cmd, flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("x", encoding="utf-8") as handle:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,
                                env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        if proc.wait():
            raise RuntimeError(f"Script thất bại; giữ output và xem {log_path}")

## 2. Dữ liệu văn bản và cấu hình đã đăng ký

Ô này tìm tệp `.jsonl.gz` trong các dataset đã gắn, kiểm tra số mẫu từng split khớp với dữ liệu đã xử lý của Task 4.0, rồi in cấu hình đã chốt trước khi chạy.

Nếu chưa có tệp, chạy trên máy cá nhân:

```
python scripts/build_text_dataset.py --out data/interim/diversevul_text.jsonl.gz
```

rồi đẩy tệp kết quả lên Kaggle thành một Dataset riêng và gắn vào notebook này.

In [ ]:
if ACTIVE:
    from src.data.text_dataset import check_split_sizes, load_text_dataset, split_sizes

    candidates = sorted(Path("/kaggle/input").glob("*/*.jsonl.gz"))
    if not candidates:
        raise FileNotFoundError(
            "Không thấy tệp .jsonl.gz nào trong /kaggle/input. "
            "Chạy scripts/build_text_dataset.py rồi gắn kết quả làm Kaggle Dataset.")
    if len(candidates) > 1:
        raise RuntimeError(f"Có nhiều hơn một tệp .jsonl.gz, chọn thủ công: {candidates}")
    TEXT_DATASET = candidates[0]
    samples = load_text_dataset(TEXT_DATASET)
    sizes = split_sizes(samples)
    problems = check_split_sizes(samples)
    print("Tệp:", TEXT_DATASET)
    print(f"Tổng {len(samples)} mẫu")
    for protocol, counts in sizes.items():
        print("  " + protocol + ": " + ", ".join(f"{k}={v}" for k, v in counts.items()))
    if problems:
        print("CẢNH BÁO tập văn bản lệch khỏi dữ liệu đã xử lý của Task 4.0:")
        for line in problems:
            print("  " + line)
    else:
        print("Khớp đúng tập mẫu mà các model graph đã dùng")

    sidecar = TEXT_DATASET.with_suffix(TEXT_DATASET.suffix + ".meta.json")
    if sidecar.is_file():
        print("Dấu vân tay kèm theo:", json.loads(sidecar.read_text(encoding="utf-8")))

    print()
    print("Cấu hình đã đăng ký trước khi chạy:")
    for name in MODELS_RUN:
        print(" ", name, "->", tx.configuration(name))
    print()
    print("Giả thuyết:", tx.HYPOTHESIS)
    print()
    print(f"Số lần huấn luyện: {len(MODELS_RUN)} model x 2 protocol x 3 seed "
          f"= {len(MODELS_RUN) * 6}")

## 3. Độ dài hàm theo token

Phải xem trước khi chạy. Hai đoạn 512 token chứa được 1020 token thân. Tỷ lệ hàm vượt mức này là phần dữ liệu mà mô hình text không nhìn thấy, và là giới hạn phải ghi vào báo cáo.

In [ ]:
if ACTIVE:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    from src.models.text_encoder import build_tokenizer, length_statistics, tokenize_samples

    FIG_DIR.mkdir(parents=True, exist_ok=True)
    length_rows, encoded_by_model = [], {}
    for name in MODELS_RUN:
        config = tx.configuration(name)
        tokenizer = build_tokenizer(config["model_name"])
        encoded = tokenize_samples(samples, tokenizer, config["max_length"], config["max_chunks"])
        encoded_by_model[name] = encoded
        for chunks in (1, config["max_chunks"], 4):
            stats = length_statistics(encoded, config["max_length"], chunks)
            length_rows.append({"model": name, "max_chunks": chunks,
                                "sức chứa token": stats["capacity_tokens"],
                                "bị cắt bỏ một phần": stats["share_truncated"],
                                "token giữ lại": stats["token_coverage"]})
    lengths = pd.DataFrame(length_rows)
    base = length_statistics(encoded_by_model[MODEL], tx.BASELINE["max_length"],
                             tx.BASELINE["max_chunks"])
    print(f"Độ dài hàm theo token ({MODEL}): trung vị {base['median_tokens']}, "
          f"p90 {base['p90_tokens']}, p99 {base['p99_tokens']}, tối đa {base['max_tokens']}")
    print(f"Hàm cần nhiều hơn một đoạn: {base['share_over_one_chunk']:.1%}")
    lengths.to_csv(FIG_DIR / "length_budget.csv", index=False)
    display(lengths.style.format({"bị cắt bỏ một phần": "{:.1%}", "token giữ lại": "{:.1%}"}))

    values = np.array([item["n_body_tokens"] for item in encoded_by_model[MODEL]])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].hist(np.clip(values, 0, 4000), bins=60, color="#5B5BD6")
    for chunks, style in ((1, "--"), (tx.BASELINE["max_chunks"], "-")):
        edge = (tx.BASELINE["max_length"] - 2) * chunks
        axes[0].axvline(edge, color="#C4320A", linestyle=style,
                        label=f"{chunks} đoạn = {edge} token")
    axes[0].set_title(f"Phân bố độ dài hàm theo token ({MODEL}), cắt hiển thị ở 4000")
    axes[0].set_xlabel("Số token thân hàm")
    axes[0].set_ylabel("Số hàm")
    axes[0].legend()
    grid = np.arange(1, 7)
    covered = [(values <= (tx.BASELINE["max_length"] - 2) * c).mean() for c in grid]
    axes[1].plot(grid, covered, marker="o", color="#118D8B")
    axes[1].axvline(tx.BASELINE["max_chunks"], color="#C4320A", linestyle="-",
                    label="cấu hình đã đăng ký")
    axes[1].set_ylim(0, 1.02)
    axes[1].set_title("Tỷ lệ hàm vào trọn trong ngân sách đoạn")
    axes[1].set_xlabel("Số đoạn tối đa")
    axes[1].set_ylabel("Tỷ lệ hàm không bị cắt")
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "length_budget.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Fine-tune

In [ ]:
if RUN_EXPERIMENTS:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for name in MODELS_RUN:
        target = OUT_DIR / name
        if target.exists() and any(target.iterdir()):
            print(f"Bỏ qua {name}, đã có kết quả")
            continue
        run_script("scripts/train_text.py",
                   ["--dataset", str(TEXT_DATASET), "--out-dir", str(target),
                    "--dataset-id", TEXT_DATASET.parent.name, *tx.arguments(name)],
                   OUT_DIR / f"{name}.log")

## 5. Kết quả của baseline text

Bảng dưới gộp theo seed. Macro-F1 trên `project_wise` là con số chính.

In [ ]:
if RUN_EXPERIMENTS:
    import pandas as pd

    def read_runs(root, names):
        frames = []
        for name in names:
            path = Path(root) / name / "results.csv"
            if path.is_file():
                frame = pd.read_csv(path)
                frame["run"] = name
                frames.append(frame)
        if not frames:
            raise FileNotFoundError(f"Không thấy results.csv nào trong {root}")
        return pd.concat(frames, ignore_index=True)

    runs = read_runs(READ_ROOT, MODELS_RUN)
    summary = (runs.pivot_table(index="run", columns=["split", "protocol"],
                                values="macro_f1", aggfunc=["mean", "std"])
               .round(4))
    display(summary)

    per_seed = (runs[runs.split == "test"]
                .pivot_table(index=["run", "protocol"], columns="seed", values="macro_f1")
                .round(4))
    print("Macro-F1 test từng seed")
    display(per_seed)

    cost = (runs[runs.split == "test"].groupby(["run", "protocol"])
            .agg(epoch_tốt_nhất=("best_epoch", list), epoch_đã_chạy=("epochs_run", list),
                 giây=("train_seconds", "sum"), vram_mb=("peak_gpu_mb", "max"),
                 tham_số=("n_params", "max")))
    print("Chi phí")
    display(cost)
    total_hours = runs[runs.split == "test"].train_seconds.sum() / 3600
    print(f"Tổng thời gian huấn luyện: {total_hours:.2f} giờ")

## 6. Đặt cạnh mọi mốc của Phase 4

Các con số tham chiếu được đọc thẳng từ tệp kết quả đã commit trong repo, không chép tay, nên không lệch với báo cáo của các task trước.

In [ ]:
if RUN_EXPERIMENTS:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    reference = pd.DataFrame(tx.reference_results(REPO_DIR))
    mine = (runs[runs.split == "test"].groupby(["run", "protocol"])
            .macro_f1.agg(["mean", lambda s: s.std(ddof=0), "count"])
            .reset_index())
    mine.columns = ["label", "protocol", "macro_f1", "std", "n_seeds"]
    mine["label"] = mine["label"].map(lambda name: f"{name} fine-tune (4.5)")
    comparison = pd.concat([reference, mine], ignore_index=True)
    table = comparison.pivot(index="label", columns="protocol", values="macro_f1").round(4)
    spread = comparison.pivot(index="label", columns="protocol", values="std").round(4)
    combined = table.astype(str) + " ± " + spread.astype(str)
    combined = combined.loc[list(dict.fromkeys(comparison.label))]
    display(combined)
    comparison.to_csv(FIG_DIR / "comparison_all_tasks.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    for axis, protocol in zip(axes, ("project_wise", "seen_project")):
        part = comparison[comparison.protocol == protocol]
        part = part.set_index("label").loc[list(dict.fromkeys(comparison.label))]
        colors = ["#C4320A" if "(4.5)" in name else
                  "#118D8B" if "token" in name or "majority" in name or "random" in name
                  else "#5B5BD6" for name in part.index]
        axis.barh(range(len(part)), part.macro_f1, xerr=part["std"], color=colors,
                  error_kw={"ecolor": "black", "capsize": 3})
        axis.set_yticks(range(len(part)))
        axis.set_yticklabels(part.index, fontsize=9)
        axis.invert_yaxis()
        axis.set_title(f"Macro-F1 test, {protocol}")
        axis.set_xlabel("Macro-F1 test, trung bình ± độ lệch chuẩn theo seed")
        axis.grid(axis="x", alpha=0.3)
    fig.suptitle("Baseline text đặt cạnh mọi mốc của Phase 4 (đỏ là 4.5, xanh lá là baseline không dùng graph)")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "comparison_all_tasks.png", dpi=150, bbox_inches="tight")
    plt.show()

## 7. Per-CWE và nhóm lớp đầu giữa đuôi

Chia nhóm theo đúng ngưỡng đã dùng ở Task 4.3 để so sánh được: đầu từ 50 mẫu test trở lên, giữa từ 16 đến 49, đuôi từ 15 trở xuống.

In [ ]:
if RUN_EXPERIMENTS:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd

    def read_per_class(root, names):
        frames = []
        for name in names:
            path = Path(root) / name / "per_class.csv"
            if path.is_file():
                frame = pd.read_csv(path)
                frame["run"] = name
                frames.append(frame)
        return pd.concat(frames, ignore_index=True)

    per_class = read_per_class(READ_ROOT, MODELS_RUN)
    focus = per_class[(per_class.split == "test") & (per_class.protocol == "project_wise")]
    by_cwe = focus.groupby(["run", "cwe"]).agg(f1=("f1", "mean"), support=("support", "max")).reset_index()
    support = by_cwe.groupby("cwe").support.max()

    def group_of(n):
        return "đầu (≥50)" if n >= 50 else ("giữa (16-49)" if n >= 16 else "đuôi (≤15)")

    by_cwe["nhóm"] = by_cwe.support.map(group_of)
    groups = (by_cwe.pivot_table(index="nhóm", columns="run", values="f1", aggfunc="mean")
              .join(by_cwe[by_cwe.run == MODEL].groupby("nhóm").size().rename("số CWE")).round(4))
    print("F1 trung bình theo nhóm lớp, test project_wise")
    display(groups)
    groups.to_csv(FIG_DIR / "per_cwe_groups.csv")

    zero = focus.groupby(["run", "seed"]).apply(
        lambda part: int((part.f1 == 0).sum()), include_groups=False)
    print("Số CWE có F1 test bằng 0 trên 23, theo từng seed:")
    print(zero.to_string())

    order = support.sort_values(ascending=False).index.tolist()
    fig, axis = plt.subplots(figsize=(14, 5))
    width = 0.8 / max(len(MODELS_RUN), 1)
    positions = np.arange(len(order))
    for index, name in enumerate(MODELS_RUN):
        values = by_cwe[by_cwe.run == name].set_index("cwe").reindex(order).f1.fillna(0)
        axis.bar(positions + index * width, values, width, label=f"{name} (4.5)")
    axis.set_xticks(positions + width * (len(MODELS_RUN) - 1) / 2)
    axis.set_xticklabels(order, rotation=90)
    axis.set_ylabel("F1 theo CWE, trung bình 3 seed")
    axis.set_title("F1 theo CWE trên test project_wise, sắp theo số mẫu giảm dần")
    twin = axis.twinx()
    twin.plot(positions + width * (len(MODELS_RUN) - 1) / 2, support.reindex(order).values,
              color="black", marker="o", markersize=3, linewidth=1)
    twin.set_ylabel("Số mẫu test")
    axis.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "per_cwe.png", dpi=150, bbox_inches="tight")
    plt.show()
    by_cwe.to_csv(FIG_DIR / "per_cwe.csv", index=False)

## 8. Đường học

Theo dõi học thuộc: train loss giảm trong khi Macro-F1 validation đứng yên là dấu hiệu mô hình ghi nhớ tập train.

In [ ]:
if RUN_EXPERIMENTS:
    import matplotlib.pyplot as plt
    import pandas as pd

    histories = []
    for name in MODELS_RUN:
        path = Path(READ_ROOT) / name / "training_history.json"
        if not path.is_file():
            continue
        for entry in json.loads(path.read_text(encoding="utf-8")):
            protocol, model_name, seed = entry["tag"].split("|")
            for step in entry["history"]:
                histories.append({"run": name, "protocol": protocol, "seed": int(seed), **step})
    history = pd.DataFrame(histories)

    fig, axes = plt.subplots(len(MODELS_RUN), 2, figsize=(13, 4 * len(MODELS_RUN)), squeeze=False)
    for row, name in enumerate(MODELS_RUN):
        for column, protocol in enumerate(("project_wise", "seen_project")):
            axis = axes[row][column]
            part = history[(history.run == name) & (history.protocol == protocol)]
            for seed, group in part.groupby("seed"):
                axis.plot(group.epoch, group.train_loss, marker="o", markersize=3,
                          label=f"seed {seed} train loss")
                axis.plot(group.epoch, group.val_macro_f1, marker="s", markersize=3,
                          linestyle="--", label=f"seed {seed} val Macro-F1")
            axis.set_title(f"{name}, {protocol}")
            axis.set_xlabel("Epoch")
            axis.grid(alpha=0.3)
            if row == 0 and column == 0:
                axis.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "learning_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    history.to_csv(FIG_DIR / "learning_curves.csv", index=False)

## 9. Ma trận nhầm lẫn

Seed đầu tiên, test, cả hai protocol. Dùng để xem mô hình có dồn dự đoán vào vài lớp nhiều mẫu không.

In [ ]:
if RUN_EXPERIMENTS:
    import matplotlib.pyplot as plt
    import numpy as np

    from src.data.text_dataset import labels_from_mapping, load_cwe_mapping

    labels = labels_from_mapping(load_cwe_mapping(REPO_DIR / "configs" / "graph" / "schema_spec.yaml"))
    panels = []
    for name in MODELS_RUN:
        path = Path(READ_ROOT) / name / "confusion_matrices.json"
        if not path.is_file():
            continue
        matrices = json.loads(path.read_text(encoding="utf-8"))
        for key, matrix in matrices.items():
            protocol, _, _, split = key.split("|")
            if split == "test":
                panels.append((f"{name}, {protocol}", np.array(matrix)))

    if panels:
        fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 6.5), squeeze=False)
        for axis, (title, matrix) in zip(axes[0], panels):
            normalised = matrix / np.clip(matrix.sum(axis=1, keepdims=True), 1, None)
            image = axis.imshow(normalised, cmap="magma", vmin=0, vmax=1)
            axis.set_title(f"{title}, chuẩn hóa theo hàng")
            axis.set_xlabel("Dự đoán")
            axis.set_ylabel("Thật")
            axis.set_xticks(range(len(labels)))
            axis.set_xticklabels(labels, rotation=90, fontsize=7)
            axis.set_yticks(range(len(labels)))
            axis.set_yticklabels(labels, fontsize=7)
            fig.colorbar(image, ax=axis, fraction=0.046)
        fig.tight_layout()
        fig.savefig(FIG_DIR / "confusion_matrices.png", dpi=150, bbox_inches="tight")
        plt.show()

## 10. Đóng gói để nộp

Gom kết quả, log và hình vào một tệp zip. Trọng số mô hình không được đưa vào vì quá nặng và không cần cho báo cáo.

In [ ]:
if RUN_EXPERIMENTS:
    if SUBMISSION_DIR.exists():
        shutil.rmtree(SUBMISSION_DIR)
    (SUBMISSION_DIR / "runs").mkdir(parents=True)
    for name in MODELS_RUN:
        source = Path(READ_ROOT) / name
        if source.is_dir():
            shutil.copytree(source, SUBMISSION_DIR / "runs" / name)
        log = Path(READ_ROOT) / f"{name}.log"
        if log.is_file():
            shutil.copy2(log, SUBMISSION_DIR / "runs" / f"{name}.log")
    shutil.copytree(FIG_DIR, SUBMISSION_DIR / "figures", dirs_exist_ok=True)
    archive = shutil.make_archive(str(SUBMISSION_DIR), "zip", SUBMISSION_DIR)
    size_mb = Path(archive).stat().st_size / 2**20
    print(f"Đã tạo {archive} ({size_mb:.1f} MB)")
    for path in sorted(SUBMISSION_DIR.rglob("*")):
        if path.is_file():
            print("  " + str(path.relative_to(SUBMISSION_DIR)))